# XOR & Neural Networks

In [4]:
import numpy as np
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

targets = {
    "AND": np.array([[0],[0],[0],[1]]),
    "OR":  np.array([[0],[1],[1],[1]]),
    "XOR": np.array([[0],[1],[1],[0]]),
} 
y = np.array([[0], [1], [1], [0]])

In [5]:
# activation functions

def step(x):
    return (x >= 0).astype(int)

# σ(x) = 1 / (1 + e^(-x))
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
 
def sigmoid_derivative(x):
    # Derived from the sigmoid: σ(x) * (1 - σ(x))
    s = sigmoid(x)
    return s * (1 - s)

In [8]:
learning_rate = 0.1
epochs = 10000
np.random.seed(1)

## Single perceptron

In [11]:
W_p = np.random.randn(2, 1) * 0.1
b_p = np.zeros((1, 1))
W_p, b_p

(array([[-0.05281718],
        [-0.10729686]]),
 array([[0.]]))

In [13]:
X @ W_p + b_p

array([[ 0.        ],
       [-0.10729686],
       [-0.05281718],
       [-0.16011404]])

In [ ]:
W_p = np.random.randn(2, 1) * 0.1
b_p = np.zeros((1, 1))

for epoch in range(epochs):
    z    = X @ W_p + b_p
    pred = step(z)
    for i in range(len(X)):
        error  = y[i] - pred[i]
        W_p   += learning_rate * error * X[i].reshape(-1, 1)
        b_p   += learning_rate * error
 
pred = step(X @ W_p + b_p)
 
print(f"\n  {'A':>2} {'B':>2} | {'Expected':>8} {'Predicted':>10}")
print(f"  {'-'*30}")
for i in range(4):
    a, b = X[i]
    exp, got = int(y[i][0]), int(pred[i][0])
    print(f"  {a:>2} {b:>2} | {exp:>8} {got:>10}")


   A  B | Expected  Predicted
  ------------------------------
   0  0 |        0          1
   0  1 |        1          1
   1  0 |        1          1
   1  1 |        0          1


## Multilayer perceptron

In [30]:
np.random.seed(1)

# layer 1: 2 inputs → 2 hidden neurons
W1 = np.random.randn(2, 2)
b1 = np.zeros((1, 2))

# layer 2: 2 hidden neurons → 1 output
W2 = np.random.randn(2, 1)
b2 = np.zeros((1, 1))

# training
for epoch in range(epochs):
    # forward pass

    # hidden layer
    z1 = X @ W1 + b1 # linear combination
    a1 = sigmoid(z1) # apply sigmoid

    # output layer
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)

    # loss: mean squared error
    loss = np.mean((a2 - y)**2)

    # backpropagation

    # gradient of loss w.r.t. output a2
    dL_da2 = 2 * (a2 - y) / y.shape[0] # MSE derivative

    # output layer gradients
    dL_dz2 = dL_da2 * sigmoid_derivative(z2)    # chain rule
    dL_dW2 = a1.T @ dL_dz2                      # gradient for W2
    dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

    # hidden layer gradients
    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * sigmoid_derivative(z1)    # chain rule 
    dL_dW1 = X.T @ dL_dz1                       # gradient for W1
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    W2 -= learning_rate * dL_dW2
    b2 -= learning_rate * dL_db2
    W1 -= learning_rate * dL_dW1
    b1 -= learning_rate * dL_db1

    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1:5d} | Loss: {loss:.6f}")


print(f"{'A':>4} {'B':>4} | {'Expected':>9} {'Predicted':>10} {'Rounded':>8}")
print("-" * 42)
for i in range(4):
    a, b      = X[i]
    expected  = int(y[i][0])
    predicted = float(a2[i][0])
    rounded   = int(round(predicted))
    print(f"{a:>4} {b:>4} | {expected:>9} {predicted:>10.4f} {rounded:>8}")

Epoch  1000 | Loss: 0.225711
Epoch  2000 | Loss: 0.202383
Epoch  3000 | Loss: 0.187222
Epoch  4000 | Loss: 0.175880
Epoch  5000 | Loss: 0.163717
Epoch  6000 | Loss: 0.152544
Epoch  7000 | Loss: 0.144523
Epoch  8000 | Loss: 0.138855
Epoch  9000 | Loss: 0.131637
Epoch 10000 | Loss: 0.083994
   A    B |  Expected  Predicted  Rounded
------------------------------------------
   0    0 |         0     0.1070        0
   0    1 |         1     0.6847        1
   1    0 |         1     0.8231        1
   1    1 |         0     0.4403        0


In [32]:
def xor_neural(a, b):
    x = np.array([[a, b]])
    h = sigmoid(x @ W1 + b1)
    out = sigmoid(h @ W2 + b2)
    return int(round(float(out[0][0])))
 
 
print("\n--- Testing xor_neural() ---")
for a, b in [(0,0), (0,1), (1,0), (1,1)]:
    print(f"xor_neural({a}, {b}) = {xor_neural(a, b)}")


--- Testing xor_neural() ---
xor_neural(0, 0) = 0
xor_neural(0, 1) = 1
xor_neural(1, 0) = 1
xor_neural(1, 1) = 0
